# RAG (Retrieval-Augmented Generation) Tutorial — RHOAI LlamaStack

This notebook is the **RHOAI variant** of the RAG tutorial. It uses the same RAG flow as `rag-tutorial.ipynb` but connects to **Red Hat OpenShift AI (RHOAI) LlamaStack** (e.g. `copilot-llama-stack`) for chat instead of the nemo-instances LlamaStack.

Full documentation: [NeMo Data Store](https://docs.nvidia.com/nemo/microservices/latest/datastore/overview.html), [NeMo Entity Store](https://docs.nvidia.com/nemo/microservices/latest/entity-store/overview.html)

## Overview

This example implements the same complete RAG workflow:
1. **Document Ingestion**: Upload documents to NeMo Data Store
2. **Embedding Generation**: Create embeddings using NeMo Embedding NIM
3. **Vector Storage**: Store embeddings in NeMo Entity Store
4. **Query Processing**: Retrieve relevant documents based on user queries
5. **Response Generation**: Generate answers using **RHOAI LlamaStack** (copilot-llama-stack) with retrieved context
6. **Optional Guardrails**: Apply safety guardrails to responses

**Chat model**: Use when your chat model is served via RHOAI's copilot-llama-stack (model id e.g. `vllm-inference/redhataillama-31-8b-instruct`). No client API key is required for copilot-llama-stack.

**Version**: RHOAI 3.2 uses Llama Stack server **0.3.5+rhai0**; this notebook uses a 0.3.x-compatible Python client.


## Prerequisites

### Deployed Services
- NeMo Data Store, Entity Store, Guardrails (optional) deployed
- **RHOAI LlamaStack** (copilot-llama-stack) deployed and running — see `deploy/rhoai/README.md`
- **Chat model** served via RHOAI (e.g. KServe InferenceService); model id like `vllm-inference/redhataillama-31-8b-instruct`
- **Embedding NIM**: `nv-embedqa-1b-v2` service

This notebook sets `LLAMASTACK_URL` and `LLAMASTACK_CHAT_MODEL` to RHOAI defaults; override in `env.donotcommit` if your namespace or model differs. **No LlamaStack API key** is used (copilot-llama-stack has no client auth).

### 🔒 Security Setup (REQUIRED FIRST STEP)

**IMPORTANT**: This notebook uses `env.donotcommit` for sensitive configuration (e.g. `NMS_NAMESPACE`, `NIM_SERVICE_ACCOUNT_TOKEN` for embeddings). 

**Before running:**
1. Copy the template: `cp env.donotcommit.example env.donotcommit`
2. Edit `env.donotcommit` and set `NMS_NAMESPACE` (and `NIM_SERVICE_ACCOUNT_TOKEN` for embeddings)
3. Optionally set `LLAMASTACK_URL` and `LLAMASTACK_CHAT_MODEL` if not using defaults

**Find your namespace:** `oc projects`


In [ ]:
# ============================================================================
# CONFIGURATION: Load Environment Variables from env.donotcommit file
# ============================================================================
# 🔒 SECURITY: Never hardcode secrets in notebooks!
# All sensitive values (tokens, API keys) should be in env.donotcommit file
# 
# SETUP INSTRUCTIONS:
# 1. Copy env.donotcommit.example to env.donotcommit: cp env.donotcommit.example env.donotcommit
# 2. Edit env.donotcommit and fill in your values (especially NMS_NAMESPACE)
# 3. env.donotcommit is git-ignored and will NOT be committed to version control
#
# IMPORTANT: Run this cell FIRST before importing config!
# If you get connection errors, restart the kernel and run cells in order.
import os
import sys
from pathlib import Path



In [ ]:
# Load env.donotcommit file from the notebook directory
try:
    from dotenv import load_dotenv
    # Find env.donotcommit file in the same directory as this notebook
    notebook_dir = Path().resolve()  # Current working directory (where notebook is run from)
    env_file = notebook_dir / "env.donotcommit"
    
    if env_file.exists():
        load_dotenv(env_file, override=False)  # override=False: don't overwrite existing env vars
        print(f"✅ Loaded env.donotcommit file from: {env_file}")
    else:
        print(f"⚠️  env.donotcommit file not found at: {env_file}")
        print(f"   Looking for env.donotcommit.example template...")
        # Check if env.donotcommit.example exists
        env_example = notebook_dir / "env.donotcommit.example"
        if env_example.exists():
            print(f"   ℹ️  env.donotcommit.example exists at: {env_example}")
            print(f"   📝 Please copy it to env.donotcommit and fill in your values:")
            print(f"      cp env.donotcommit.example env.donotcommit")
            print(f"      # Then edit env.donotcommit and add your NMS_NAMESPACE")
        else:
            print(f"   ⚠️  env.donotcommit.example not found - creating template...")
            env_example_content = """# NeMo Microservices Configuration
# Copy this file to env.donotcommit and fill in your values
# env.donotcommit is git-ignored and will NOT be committed

# REQUIRED: Namespace for cluster services
# Replace with your actual OpenShift namespace/project name
# Find your namespace: oc projects
NMS_NAMESPACE=your-namespace



# OPTIONAL: NeMo Data Store token
# Default is "token" - update if your deployment uses a different token
NDS_TOKEN=token

# OPTIONAL: Dataset name for RAG tutorial documents
DATASET_NAME=rag-tutorial-documents

# OPTIONAL: RAG Configuration
# Number of documents to retrieve
RAG_TOP_K=5
# Similarity threshold for retrieval
RAG_SIMILARITY_THRESHOLD=0.3

# OPTIONAL: API Keys (only needed if using external APIs as fallback)
# OPENAI_API_KEY=
# NVIDIA_API_KEY=
# HF_TOKEN=
"""
            env_example.write_text(env_example_content)
            print(f"   ✅ Created env.donotcommit.example template at: {env_example}")
            print(f"   📝 Please copy it to env.donotcommit and fill in your values:")
            print(f"      cp env.donotcommit.example env.donotcommit")
            print(f"      # Then edit env.donotcommit and add your NMS_NAMESPACE")
except ImportError:
    print("⚠️  python-dotenv not installed - install with: pip install python-dotenv")
    print("   Will use system environment variables only (not recommended)")

In [ ]:
# Clear any cached config module to force reload
if 'config' in sys.modules:
    del sys.modules['config']
    print("⚠️  Cleared cached config module - will reload with new env vars")

# Avoid tokenizers fork warning (e.g. with sentence-transformers)
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

# Set defaults (override with env.donotcommit if present)
ns = os.environ.get("NMS_NAMESPACE", "anemo-rhoai")
os.environ.setdefault("NMS_NAMESPACE", ns)
os.environ.setdefault("NDS_TOKEN", "token")
os.environ.setdefault("DATASET_NAME", "rag-tutorial-documents")
os.environ.setdefault("RAG_TOP_K", "5")
os.environ.setdefault("RAG_SIMILARITY_THRESHOLD", "0.3")

# RHOAI LlamaStack defaults (copilot-llama-stack) — override in env.donotcommit if needed
os.environ.setdefault("LLAMASTACK_URL", f"http://copilot-llama-stack-service.{ns}.svc.cluster.local:8321")
os.environ.setdefault("LLAMASTACK_CHAT_MODEL", "vllm-inference/redhataillama-31-8b-instruct")
os.environ.setdefault("LLAMASTACK_API_KEY", "")  # No client auth for RHOAI copilot-llama-stack

print("\n✅ Environment variables loaded")
print(f"   NMS_NAMESPACE: {os.environ.get('NMS_NAMESPACE')}")
print(f"   DATASET_NAME: {os.environ.get('DATASET_NAME')}")
print(f"   LlamaStack (RHOAI): {os.environ.get('LLAMASTACK_URL')}")
print(f"   Chat model: {os.environ.get('LLAMASTACK_CHAT_MODEL')}")
print(f"   Mode: Cluster (Workbench/Notebook)")
print(f"\n💡 If you see connection errors, restart the kernel and run cells in order!")

In [ ]:
# Install llama-stack-client compatible with RHOAI LlamaStack server 0.3.5.x
# Do NOT use main/git — RHOAI server is 0.3.5.1; client 0.5.x returns 426 version mismatch
%pip install "llama-stack-client>=0.3,<0.4"


In [ ]:
# Install required packages (llama-stack-client already pinned in previous cell for RHOAI)
%pip install requests jupyterlab python-dotenv numpy pandas huggingface_hub sentence-transformers


In [ ]:
# Load configuration
from config import (
    NDS_URL, ENTITY_STORE_URL, GUARDRAILS_URL,
    NIM_CHAT_URL, NIM_EMBEDDING_URL,
    NIM_CHAT_URL_CLUSTER, NIM_EMBEDDING_URL_CLUSTER,
    NMS_NAMESPACE, DATASET_NAME, NDS_TOKEN,
    RAG_TOP_K, RAG_SIMILARITY_THRESHOLD,
    LLAMASTACK_URL, LLAMASTACK_CHAT_MODEL, LLAMASTACK_API_KEY, NIM_SERVICE_ACCOUNT_TOKEN,
    validate_config,
)
validate_config()  # Fail fast if required env (e.g. NIM_CHAT_URL) is missing

print(f"✅ Configuration loaded (RHOAI LlamaStack variant)")
print(f"Mode: Cluster (Workbench/Notebook)")
print(f"Data Store: {NDS_URL}")
print(f"Entity Store: {ENTITY_STORE_URL}")
print(f"LlamaStack (RHOAI): {LLAMASTACK_URL}")
print(f"LlamaStack chat model: {LLAMASTACK_CHAT_MODEL}")
print(f"Namespace: {NMS_NAMESPACE}")
print(f"Dataset: {DATASET_NAME}")


In [ ]:
# Quick connectivity test
import requests
try:
    r = requests.get(f"{NDS_URL}/v1/datastore/namespaces", timeout=2)
    print(f"✅ Data Store connectivity: OK")
except Exception as e:
    print(f"⚠️  Data Store connectivity: FAILED - {e}")
    print(f"   Ensure you're running this notebook from within a Workbench/Notebook in the cluster")
    print(f"   Verify Data Store is running: oc get pods -n {NMS_NAMESPACE} | grep datastore")

In [ ]:
# Initialize LlamaStack client
try:
    from llama_stack_client import LlamaStackClient
    import logging
    
    # Suppress httpx INFO logs (500 errors and connection attempts are expected during fallback)
    logging.getLogger("httpx").setLevel(logging.WARNING)
    
    client = LlamaStackClient(base_url=LLAMASTACK_URL, api_key=LLAMASTACK_API_KEY or None)
    # Test connectivity
    # Note: 404 on root endpoint is expected - it just means the service is reachable
    try:
        server_info = client._client.get("/")
        print(f"✅ LlamaStack connectivity: OK")
        try:
            client_version = client._client._version
            print(f"   LlamaStack client version: {client_version}")
        except:
            pass
    except Exception as e:
        # 404 is OK - it means service is reachable but root endpoint doesn't exist
        if "404" in str(e) or "Not Found" in str(e):
            print(f"✅ LlamaStack connectivity: OK (service reachable)")
        else:
            print(f"⚠️  LlamaStack connectivity: FAILED - {e}")
            print(f"   Make sure RHOAI LlamaStack (copilot-llama-stack) is deployed: oc get pods -n {NMS_NAMESPACE} | grep llama")
            client = None
except ImportError:
    print("⚠️  LlamaStack client not available - install with: %pip install \"llama-stack-client>=0.3,<0.4\"")
    print("   Continuing without LlamaStack integration...")
    client = None
except Exception as e:
    print(f"⚠️  LlamaStack initialization failed: {e}")
    print("   Continuing without LlamaStack integration...")
    client = None

## Step 1: Document Ingestion

First, we'll upload sample documents to NeMo Data Store. These documents will be used for retrieval.


In [ ]:
# Sample documents for RAG tutorial
documents = [
    {
        "id": "doc1",
        "title": "Introduction to NeMo Microservices",
        "content": "NVIDIA NeMo Microservices is a platform for deploying AI models at scale. It provides infrastructure for training, inference, and evaluation of large language models. The platform includes components like Data Store, Entity Store, Customizer, Evaluator, and Guardrails."
    },
    {
        "id": "doc2",
        "title": "RAG Architecture",
        "content": "Retrieval-Augmented Generation (RAG) combines information retrieval with language generation. The process involves: 1) Storing documents in a vector database, 2) Embedding user queries, 3) Retrieving relevant documents, 4) Generating responses using retrieved context."
    },
    {
        "id": "doc3",
        "title": "OpenShift Deployment",
        "content": "NeMo Microservices can be deployed on OpenShift using Helm charts. The deployment includes infrastructure components (PostgreSQL, MLflow, Argo Workflows) and instance components (NeMo services, NIM services). All components are namespace-scoped for multi-tenant safety."
    },
    {
        "id": "doc4",
        "title": "NIM Services",
        "content": "NVIDIA Inference Microservices (NIM) provide optimized inference for AI models. NIM services support chat models, embedding models, and reranking models. They are containerized and can be deployed on Kubernetes/OpenShift clusters with GPU support."
    },
    {
        "id": "doc5",
        "title": "Vector Databases",
        "content": "Vector databases store embeddings for similarity search. NeMo Entity Store provides vector storage capabilities. Milvus is also available as an alternative vector database. Both support efficient similarity search for RAG applications."
    }
]

print(f"✅ Prepared {len(documents)} sample documents")
for doc in documents:
    print(f"  - {doc['title']}")


In [ ]:
# Upload documents to NeMo Data Store
# IMPORTANT: For LlamaStack dataset registration to work, files must be uploaded to Data Store FIRST
# Then we can register the dataset with LlamaStack
import json
import tempfile
import os

print("Step 1: Creating namespace in Data Store...")

# Create namespace if it doesn't exist
namespace_url = f"{NDS_URL}/v1/datastore/namespaces/{NMS_NAMESPACE}"
try:
    response = requests.get(namespace_url, headers={"Authorization": f"Bearer {NDS_TOKEN}"})
    if response.status_code == 404:
        # Create namespace
        response = requests.post(
            f"{NDS_URL}/v1/datastore/namespaces",
            json={"name": NMS_NAMESPACE},
            headers={"Authorization": f"Bearer {NDS_TOKEN}"}
        )
        print(f"✅ Created namespace: {NMS_NAMESPACE}")
    else:
        print(f"✅ Namespace exists: {NMS_NAMESPACE}")
except Exception as e:
    print(f"⚠️  Error checking namespace: {e}")


In [ ]:
# Step 2: Upload files to Data Store using HuggingFace API
print(f"\nStep 2: Uploading documents to Data Store...")
uploaded_docs = []
repo_id = f"{NMS_NAMESPACE}/{DATASET_NAME}"

try:
    from huggingface_hub import HfApi
    
    # Initialize HuggingFace API pointing to Data Store
    hf_endpoint = f"{NDS_URL}/v1/hf"
    hf_token = NDS_TOKEN if NDS_TOKEN != "token" else None
    hf_api = HfApi(endpoint=hf_endpoint, token=hf_token)
    
    # Create dataset repo in Data Store
    try:
        hf_api.create_repo(repo_id=repo_id, repo_type="dataset", exist_ok=True)
        print(f"✅ Created dataset repo: {repo_id}")
    except Exception as e:
        if "already exists" in str(e).lower():
            print(f"ℹ️  Dataset repo already exists: {repo_id}")
        else:
            print(f"⚠️  Error creating repo: {e}")
            raise
    
    # Upload each document as a JSON file
    for doc in documents:
        try:
            doc_data = {
                "id": doc['id'],
                "title": doc['title'],
                "content": doc['content']
            }
            
            # Create temporary file
            with tempfile.NamedTemporaryFile(mode='w', suffix='.json', delete=False) as f:
                json.dump(doc_data, f, indent=2)
                temp_file = f.name
            
            try:
                # Upload file to Data Store
                file_path = f"{doc['id']}.json"
                hf_api.upload_file(
                    path_or_fileobj=temp_file,
                    path_in_repo=file_path,
                    repo_id=repo_id,
                    repo_type="dataset"
                )
                uploaded_docs.append(doc_data)
                print(f"✅ Uploaded: {doc['title']} -> {file_path}")
            finally:
                os.unlink(temp_file)
                
        except Exception as e:
            print(f"⚠️  Error uploading {doc['id']}: {e}")
    
    print(f"\n✅ Uploaded {len(uploaded_docs)} documents to Data Store")
    
except ImportError:
    print("⚠️  huggingface_hub not installed - skipping file upload")
    print("   Install with: pip install huggingface_hub")
    print("   For this tutorial, we'll use in-memory documents instead")
    # Fallback: use in-memory documents
    uploaded_docs = [{"id": doc['id'], "title": doc['title'], "content": doc['content']} 
                     for doc in documents]
    print(f"✅ Prepared {len(uploaded_docs)} documents in memory")
    repo_id = None  # Mark that we didn't upload to Data Store
    
except Exception as e:
    print(f"⚠️  Error uploading to Data Store: {e}")
    print("   For this tutorial, we'll use in-memory documents instead")
    # Fallback: use in-memory documents
    uploaded_docs = [{"id": doc['id'], "title": doc['title'], "content": doc['content']} 
                     for doc in documents]
    print(f"✅ Prepared {len(uploaded_docs)} documents in memory")
    repo_id = None  # Mark that we didn't upload to Data Store

In [ ]:
# Step 3: Register dataset with LlamaStack (only if files were uploaded)
# IMPORTANT: Make sure the upload cell (Step 2) completed successfully before running this cell
if repo_id and client is not None:
    if not hasattr(client, "beta"):
        print(f"\nStep 3: LlamaStack dataset registration skipped (RHOAI client 0.3.x has no beta API).")
        print(f"   Dataset registration is optional for this RAG tutorial — we only use LlamaStack for chat.")
        print(f"   Proceed to the next steps.")
    else:
        print(f"\nStep 3: Registering dataset with LlamaStack...")
        print(f"   Dataset: {repo_id}")
        print(f"   Make sure Step 2 (upload) completed successfully!")
        
        try:
            import warnings
            dataset_uri = f"hf://datasets/{repo_id}"
            print(f"   Dataset URI: {dataset_uri}")
            
            with warnings.catch_warnings():
                warnings.filterwarnings("ignore", category=DeprecationWarning, message=".*deprecated.*")
                response = client.beta.datasets.register(
                    purpose="post-training/messages",
                    dataset_id=DATASET_NAME,
                    source={
                        "type": "uri",
                        "uri": dataset_uri
                    },
                    metadata={
                        "format": "json",
                        "description": f"RAG tutorial documents for {DATASET_NAME}",
                        "provider_id": "nvidia",
                    }
                )
            print(f"✅ Dataset registered with LlamaStack: {DATASET_NAME}")
            if hasattr(response, 'dataset_id'):
                print(f"   Dataset ID: {response.dataset_id}")
        except Exception as e:
            error_msg = str(e)
            if "already exists" in error_msg.lower() or "409" in error_msg:
                print(f"ℹ️  Dataset {DATASET_NAME} already exists in LlamaStack (this is OK)")
            elif "426" in error_msg or "not compatible with server version" in error_msg.lower():
                print(f"ℹ️  LlamaStack client/server version mismatch (e.g. RHOAI server 0.3.x).")
                print(f"   Dataset registration is optional for this RAG tutorial — we only use LlamaStack for chat.")
                print(f"   Continuing without registration. You can proceed to the next steps.")
            else:
                print(f"⚠️  Error registering dataset with LlamaStack: {error_msg}")
                print(f"\n💡 Troubleshooting:")
                print(f"   1. Make sure Step 2 (upload) cell completed successfully")
                print(f"   2. Verify files were uploaded: Check the output of Step 2 cell")
                print(f"   3. Wait a few seconds and try running Step 2 again, then this cell")
                print(f"   4. Check Data Store: oc get pods -n {NMS_NAMESPACE} | grep datastore")
                print(f"   5. Check LlamaStack: oc get pods -n {NMS_NAMESPACE} | grep llamastack")
                print(f"   6. If dataset already exists, you can skip this step")
                print(f"\n   Continuing without LlamaStack registration...")
elif not repo_id:
    print(f"\n⚠️  Skipping LlamaStack registration (files not uploaded to Data Store)")
    print(f"   Run Step 2 (upload) cell first to upload files to Data Store")
elif client is None:
    print(f"\n⚠️  Skipping LlamaStack registration (LlamaStack client not available)")
    print(f"   Make sure the LlamaStack initialization cell ran successfully")
else:
    print(f"\n⚠️  Skipping LlamaStack registration")

In [ ]:
# Step 4: Register dataset in Entity Store (required for some NeMo services)
if repo_id:
    print(f"\nStep 4: Registering dataset in Entity Store...")
    try:
        response = requests.post(
            f"{ENTITY_STORE_URL}/v1/datasets",
            json={
                "name": DATASET_NAME,
                "namespace": NMS_NAMESPACE,
                "description": f"RAG tutorial documents for {DATASET_NAME}",
                "files_url": f"hf://datasets/{repo_id}",
                "project": "rag-tutorial",
                "format": "json",
            },
        )
        
        if response.status_code in (200, 201):
            print(f"✅ Dataset registered in Entity Store: {DATASET_NAME}")
            dataset_obj = response.json()
            if 'files_url' in dataset_obj:
                print(f"   Files URL: {dataset_obj['files_url']}")
        elif response.status_code == 409:
            print(f"ℹ️  Dataset {DATASET_NAME} already exists in Entity Store (this is OK)")
        else:
            print(f"⚠️  Failed to register in Entity Store: {response.status_code} - {response.text}")
    except Exception as e:
        print(f"⚠️  Error registering dataset in Entity Store: {e}")
        print(f"   Continuing without Entity Store registration...")
else:
    print(f"\n⚠️  Skipping Entity Store registration (files not uploaded to Data Store)")

print(f"\n✅ Document ingestion complete: {len(uploaded_docs)} documents ready for embedding")

## Step 2: Generate Embeddings

Now we'll generate embeddings for each document using the NeMo Embedding NIM service.


In [ ]:
# Generate embeddings using NeMo Embedding NIM (or CPU fallback when NIM has no GPU)
# If the embedding NIM pod is Pending (Insufficient nvidia.com/gpu), we fall back to sentence-transformers on CPU.
_embedding_fallback_model = None  # Lazy-loaded CPU model

def get_embedding(text, embedding_url, input_type="passage"):
    """Generate embedding for text using NeMo Embedding NIM, or CPU fallback if NIM is unreachable."""
    global _embedding_fallback_model
    # 1) Try NIM first
    try:
        headers = {"Content-Type": "application/json"}
        if NIM_SERVICE_ACCOUNT_TOKEN:
            headers["Authorization"] = f"Bearer {NIM_SERVICE_ACCOUNT_TOKEN}"
        response = requests.post(
            f"{embedding_url}/v1/embeddings",
            json={"input": text, "model": "nvidia/llama-3.2-nv-embedqa-1b-v2", "input_type": input_type},
            headers=headers,
            timeout=30
        )
        if response.status_code == 200:
            return response.json()["data"][0]["embedding"]
        # Non-200: don't fall back, let caller see the error
        print(f"⚠️  Error getting embedding: {response.status_code} - {response.text}")
        return None
    except Exception as e:
        err_str = str(e).lower()
        # 2) Connection refused / NIM pod not running (e.g. Pending for GPU) -> try CPU fallback
        if "connection refused" in err_str or "max retries" in err_str or "111" in err_str:
            if _embedding_fallback_model is None:
                try:
                    from sentence_transformers import SentenceTransformer
                    print("ℹ️  Embedding NIM unreachable (often: pod Pending — no GPU). Using CPU fallback: sentence-transformers/all-MiniLM-L6-v2")
                    _embedding_fallback_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
                except ImportError:
                    print("⚠️  NIM unreachable and sentence-transformers not installed. Install with: %pip install sentence-transformers")
                    print("   Or free a GPU so the embedding NIM pod (nv-embedqa-1b-v2) can schedule.")
                    return None
            emb = _embedding_fallback_model.encode(text, convert_to_numpy=True)
            return emb.tolist()
        print(f"⚠️  Exception getting embedding: {e}")
        return None

# Generate embeddings for all documents
print("Generating embeddings...")
documents_with_embeddings = []

for doc in uploaded_docs:
    text_to_embed = f"{doc['title']}\n{doc['content']}"
    embedding = get_embedding(text_to_embed, NIM_EMBEDDING_URL)
    if embedding:
        doc['embedding'] = embedding
        documents_with_embeddings.append(doc)
        print(f"✅ Generated embedding for: {doc['title']}")
    else:
        print(f"⚠️  Failed to generate embedding for: {doc['title']}")

print(f"\n✅ Generated embeddings for {len(documents_with_embeddings)} documents")


## Step 3: Store Embeddings Locally

For this tutorial, we'll store embeddings in memory for local similarity search.
In production, you can use a vector database like Milvus or Pinecone.

**Note**: NeMo Entity Store is primarily designed for managing models, datasets, and namespaces,
not for storing arbitrary document embeddings. For production RAG, consider using a dedicated vector database.**


In [ ]:
# Generate embeddings for all documents
print("Generating embeddings...")
documents_with_embeddings = []

for doc in uploaded_docs:
    # Combine title and content for embedding
    text_to_embed = f"{doc['title']}\n{doc['content']}"
    embedding = get_embedding(text_to_embed, NIM_EMBEDDING_URL)
    
    if embedding:
        doc['embedding'] = embedding
        documents_with_embeddings.append(doc)
        print(f"✅ Generated embedding for: {doc['title']}")
    else:
        print(f"⚠️  Failed to generate embedding for: {doc['title']}")

print(f"\n✅ Generated embeddings for {len(documents_with_embeddings)} documents")

In [ ]:
# Embeddings are already stored in documents_with_embeddings list
# For this tutorial, we use in-memory storage for simplicity
print(f"✅ Stored {len(documents_with_embeddings)} documents with embeddings in memory")
print(f"   Documents ready for local similarity search")
print(f"   In production, use a vector database like Milvus or Pinecone")


## Step 4: Query and Retrieve

Now we'll process a user query: embed it, find similar documents, and retrieve the most relevant ones.


In [ ]:
# User query
user_query = "What is RAG and how does it work?"

print(f"User Query: {user_query}\n")

# Generate embedding for the query
query_embedding = get_embedding(user_query, NIM_EMBEDDING_URL, input_type="query")

if query_embedding:
    print(f"✅ Generated query embedding (dimension: {len(query_embedding)})\n")
    
    # Use local similarity search
    import numpy as np
    retrieved_docs = []
    similarities = []
    
    for doc in documents_with_embeddings:
        similarity = np.dot(query_embedding, doc['embedding']) / (
            np.linalg.norm(query_embedding) * np.linalg.norm(doc['embedding'])
        )
        similarities.append((similarity, doc))
    
    # Sort by similarity and get top_k
    similarities.sort(reverse=True, key=lambda x: x[0])
    
    print(f"✅ Found {len(similarities)} documents, showing top {RAG_TOP_K}:\n")
    
    for i, (similarity, doc) in enumerate(similarities[:RAG_TOP_K], 1):
        if similarity >= RAG_SIMILARITY_THRESHOLD:
            print(f"{i}. {doc['title']} (similarity: {similarity:.3f})")
            retrieved_docs.append({
                'title': doc['title'],
                'content': doc['content'],
                'id': doc['id']
            })
    
    print(f"\n✅ Retrieved {len(retrieved_docs)} documents above threshold ({RAG_SIMILARITY_THRESHOLD})\n")
else:
    print("⚠️  Failed to generate query embedding")
    retrieved_docs = []


## Step 5: Generate Response

Now we'll use the retrieved documents as context to generate a response using the Chat NIM.


In [ ]:
# Build context from retrieved documents
context = "\n\n".join([
    f"Document: {doc['title']}\n{doc['content']}"
    for doc in retrieved_docs
])

print("Retrieved Context:")
print("=" * 80)
print(context[:500] + "..." if len(context) > 500 else context)
print("=" * 80)
print()

# Generate response using LlamaStack client only (no fallback)
def generate_response(query, context):
    """Generate response using LlamaStack client with retrieved context"""
    # Validate LlamaStack client is available
    if client is None:
        raise ValueError(
            "LlamaStack client not available. "
            "Check LlamaStack deployment and connectivity."
        )
    
    # Build prompt with context
    system_prompt = "You are a helpful assistant. Answer the question based on the provided context. If the context doesn't contain enough information, say so."
    user_prompt = f"Context:\n{context}\n\nQuestion: {query}\n\nAnswer:"
    
    try:
        response = client.chat.completions.create(
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            model=LLAMASTACK_CHAT_MODEL,
            temperature=0.7,
            max_tokens=500
        )
        return response.choices[0].message.content
    except Exception as e:
        error_msg = str(e)
        print(f"❌ LlamaStack error: {error_msg}")
        print("\n🔍 Troubleshooting steps:")
        print(f"1. Check LlamaStack pod is running:")
        print(f"   oc get pods -n {NMS_NAMESPACE} | grep llamastack")
        print(f"2. Check LlamaStack pod status:")
        print(f"   oc get pods -n {NMS_NAMESPACE} -l app=nemo-llamastack")
        print(f"3. Check LlamaStack logs for errors:")
        print(f"   oc logs -n {NMS_NAMESPACE} deployment/llamastack -c llamastack-ctr --tail=50")
        print(f"4. Verify token secret exists and is populated:")
        print(f"   oc get secret -n {NMS_NAMESPACE} | grep token-secret")
        print(f"   oc get secret <sa-name>-token-secret -n {NMS_NAMESPACE} -o jsonpath='{{.data.token}}' | base64 -d | head -c 20")
        print(f"5. Check service account exists:")
        print(f"   oc get sa -n {NMS_NAMESPACE} | grep model")
        print(f"6. Verify LlamaStack can reach NIM (check initContainer logs):")
        print(f"   oc logs -n {NMS_NAMESPACE} <pod-name> -c wait-for-token")
        
        # Provide specific guidance based on error type
        if "426" in error_msg or "not compatible with server version" in error_msg.lower():
            print(f"\n💡 This is a 426 error - Client/server version mismatch (RHOAI server is 0.3.5.x).")
            print(f"   Fix: Install a compatible client, then restart kernel and re-run from the top:")
            print(f"   %pip install \"llama-stack-client>=0.3,<0.4\"")
        elif "500" in error_msg or "Internal server error" in error_msg:
            print(f"\n💡 This is a 500 error - LlamaStack is having internal issues.")
            print(f"   Most likely causes:")
            print(f"   - Token secret not populated (check step 4)")
            print(f"   - LlamaStack can't authenticate to NIM")
            print(f"   - Check LlamaStack logs (step 3) for detailed error")
        elif "401" in error_msg or "Unauthorized" in error_msg:
            print(f"\n💡 This is a 401 error - Authentication failed.")
            print(f"   Most likely causes:")
            print(f"   - Token secret missing or empty")
            print(f"   - Service account doesn't exist")
            print(f"   - Check token secret (step 4)")
        elif "404" in error_msg or "Not Found" in error_msg:
            print(f"\n💡 This is a 404 error - Endpoint not found.")
            print(f"   Most likely causes:")
            print(f"   - LlamaStack not fully started")
            print(f"   - Wrong URL configured")
            print(f"   - Check pod status (step 2)")
        
        raise RuntimeError(f"LlamaStack request failed: {error_msg}")

# Generate response
print("Generating response...")
response_text = generate_response(user_query, context)

if response_text:
    print("\n" + "=" * 80)
    print("Generated Response:")
    print("=" * 80)
    print(response_text)
    print("=" * 80)
else:
    print("⚠️  Failed to generate response")


## Step 7: Validate RAG with Test Queries

Let's test the RAG pipeline with multiple questions to validate it's working correctly.
We'll ask questions about different topics from our documents and verify the responses.

In [ ]:
# Use local similarity search to find relevant documents
if query_embedding:
    import numpy as np
    retrieved_docs = []
    similarities = []
    
    for doc in documents_with_embeddings:
        similarity = np.dot(query_embedding, doc['embedding']) / (
            np.linalg.norm(query_embedding) * np.linalg.norm(doc['embedding'])
        )
        similarities.append((similarity, doc))
    
    # Sort by similarity and get top_k
    similarities.sort(reverse=True, key=lambda x: x[0])
    
    print(f"✅ Found {len(similarities)} documents, showing top {RAG_TOP_K}:\n")
    
    for i, (similarity, doc) in enumerate(similarities[:RAG_TOP_K], 1):
        if similarity >= RAG_SIMILARITY_THRESHOLD:
            print(f"{i}. {doc['title']} (similarity: {similarity:.3f})")
            retrieved_docs.append({
                'title': doc['title'],
                'content': doc['content'],
                'id': doc['id']
            })
    
    print(f"\n✅ Retrieved {len(retrieved_docs)} documents above threshold ({RAG_SIMILARITY_THRESHOLD})\n")
else:
    retrieved_docs = []

In [ ]:
# Test queries covering different topics from our documents
test_queries = [
    "What is NeMo Microservices?",
    "How does RAG work?",
    "What are vector databases used for?",
    "What components does NeMo Microservices include?"
]

print("=" * 80)
print("RAG VALIDATION TESTS")
print("=" * 80)
print(f"Testing {len(test_queries)} queries against {len(documents_with_embeddings)} documents\n")

In [ ]:
# Function to run a complete RAG query
def run_rag_query(query, show_context=True):
    """Run a complete RAG query and return the response"""
    print(f"\n{'='*80}")
    print(f"Query: {query}")
    print(f"{'='*80}")
    
    # Generate query embedding
    query_embedding = get_embedding(query, NIM_EMBEDDING_URL, input_type="query")
    
    if not query_embedding:
        print("⚠️  Failed to generate query embedding")
        return None
    
    # Local similarity search
    import numpy as np
    retrieved_docs = []
    similarities = []
    
    for doc in documents_with_embeddings:
        similarity = np.dot(query_embedding, doc['embedding']) / (
            np.linalg.norm(query_embedding) * np.linalg.norm(doc['embedding'])
        )
        similarities.append((similarity, doc))
    
    # Sort by similarity and get top_k
    similarities.sort(reverse=True, key=lambda x: x[0])
    
    print(f"\n📊 Top {min(RAG_TOP_K, len(similarities))} retrieved documents:")
    for i, (similarity, doc) in enumerate(similarities[:RAG_TOP_K], 1):
        if similarity >= RAG_SIMILARITY_THRESHOLD:
            print(f"  {i}. {doc['title']} (similarity: {similarity:.3f})")
            retrieved_docs.append({
                'title': doc['title'],
                'content': doc['content'],
                'id': doc['id']
            })
    
    if not retrieved_docs:
        print(f"⚠️  No documents found above threshold ({RAG_SIMILARITY_THRESHOLD})")
        return None
    
    # Build context
    context = "\n\n".join([
        f"Document: {doc['title']}\n{doc['content']}"
        for doc in retrieved_docs
    ])
    
    if show_context:
        print(f"\n📄 Retrieved Context (first 300 chars):")
        print(f"{context[:300]}...")
    
    # Generate response using LlamaStack client (via generate_response function)
    print(f"\n🤖 Generating response...")
    response_text = generate_response(query, context)
    
    if response_text:
        print(f"\n✅ Response:")
        print(f"{response_text}")
        return response_text
    else:
        print(f"⚠️  Failed to generate response")
        return None

In [ ]:
# Generate response using LlamaStack client only (no fallback)
def generate_response(query, context):
    """Generate response using LlamaStack client with retrieved context"""
    # Validate LlamaStack client is available
    if client is None:
        raise ValueError(
            "LlamaStack client not available. "
            "Check LlamaStack deployment and connectivity."
        )
    
    # Build prompt with context
    system_prompt = "You are a helpful assistant. Answer the question based on the provided context. If the context doesn't contain enough information, say so."
    user_prompt = f"Context:\n{context}\n\nQuestion: {query}\n\nAnswer:"
    
    try:
        response = client.chat.completions.create(
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            model=LLAMASTACK_CHAT_MODEL,
            temperature=0.7,
            max_tokens=500
        )
        return response.choices[0].message.content
    except Exception as e:
        error_msg = str(e)
        print(f"❌ LlamaStack error: {error_msg}")
        print("\n🔍 Troubleshooting steps:")
        print(f"1. Check LlamaStack pod is running:")
        print(f"   oc get pods -n {NMS_NAMESPACE} | grep llamastack")
        print(f"2. Check LlamaStack pod status:")
        print(f"   oc get pods -n {NMS_NAMESPACE} -l app=nemo-llamastack")
        print(f"3. Check LlamaStack logs for errors:")
        print(f"   oc logs -n {NMS_NAMESPACE} deployment/llamastack -c llamastack-ctr --tail=50")
        print(f"4. Verify token secret exists and is populated:")
        print(f"   oc get secret -n {NMS_NAMESPACE} | grep token-secret")
        print(f"   oc get secret <sa-name>-token-secret -n {NMS_NAMESPACE} -o jsonpath='{{.data.token}}' | base64 -d | head -c 20")
        print(f"5. Check service account exists:")
        print(f"   oc get sa -n {NMS_NAMESPACE} | grep model")
        print(f"6. Verify LlamaStack can reach NIM (check initContainer logs):")
        print(f"   oc logs -n {NMS_NAMESPACE} <pod-name> -c wait-for-token")
        
        # Provide specific guidance based on error type
        if "426" in error_msg or "not compatible with server version" in error_msg.lower():
            print(f"\n💡 This is a 426 error - Client/server version mismatch (RHOAI server is 0.3.5.x).")
            print(f"   Fix: Install a compatible client, then restart kernel and re-run from the top:")
            print(f"   %pip install \"llama-stack-client>=0.3,<0.4\"")
        elif "500" in error_msg or "Internal server error" in error_msg:
            print(f"\n💡 This is a 500 error - LlamaStack is having internal issues.")
            print(f"   Most likely causes:")
            print(f"   - Token secret not populated (check step 4)")
            print(f"   - LlamaStack can't authenticate to NIM")
            print(f"   - Check LlamaStack logs (step 3) for detailed error")
        elif "401" in error_msg or "Unauthorized" in error_msg:
            print(f"\n💡 This is a 401 error - Authentication failed.")
            print(f"   Most likely causes:")
            print(f"   - Token secret missing or empty")
            print(f"   - Service account doesn't exist")
            print(f"   - Check token secret (step 4)")
        elif "404" in error_msg or "Not Found" in error_msg:
            print(f"\n💡 This is a 404 error - Endpoint not found.")
            print(f"   Most likely causes:")
            print(f"   - LlamaStack not fully started")
            print(f"   - Wrong URL configured")
            print(f"   - Check pod status (step 2)")
        
        raise RuntimeError(f"LlamaStack request failed: {error_msg}")

In [ ]:
# Generate response
print("Generating response...")
response_text = generate_response(user_query, context)

if response_text:
    print("\n" + "=" * 80)
    print("Generated Response:")
    print("=" * 80)
    print(response_text)
    print("=" * 80)
else:
    print("⚠️  Failed to generate response")

In [ ]:
# Run all test queries
results = {}

for query in test_queries:
    response = run_rag_query(query, show_context=True)
    results[query] = response
    print("\n" + "-"*80 + "\n")

print("=" * 80)
print("VALIDATION SUMMARY")
print("=" * 80)
print(f"\nTotal queries tested: {len(test_queries)}")
print(f"Successful responses: {sum(1 for r in results.values() if r is not None)}")
print(f"Failed responses: {sum(1 for r in results.values() if r is None)}")

if all(r is not None for r in results.values()):
    print("\n✅ All queries returned responses! RAG pipeline is working correctly.")
else:
    print("\n⚠️  Some queries failed. Check the error messages above.")

## Summary

This tutorial demonstrated a complete RAG pipeline:
1. ✅ Document ingestion into NeMo Data Store
2. ✅ Embedding generation using NeMo Embedding NIM
3. ✅ Vector storage (local in-memory for this tutorial)
4. ✅ Query processing with similarity search
5. ✅ Response generation using RHOAI LlamaStack (copilot-llama-stack)
6. ✅ Optional guardrails validation
7. ✅ RAG validation with test queries

### Next Steps

- Add more documents to improve retrieval quality
- Experiment with different embedding models
- Adjust retrieval parameters (top_k, similarity threshold)
- Integrate with your own document sources
- Add multi-turn conversation support
- Use a production vector database (Milvus, Pinecone, etc.) for larger document sets